# AnalyticsHub Load Status (Databricks)

Databricks adaptation of the corresponding ValueLens notebook. It uses the configured Unity Catalog schema and volume; see the [Databricks route guide](../README.md) for setup and scheduling.


In [ ]:
from valuelens_databricks_runtime import initialize
VL = initialize(spark, dbutils)
DB_SCHEMA = VL.schema
VL.record_task_start()


In [ ]:
# === CONFIG ===
LOG_TABLE             = f'{DB_SCHEMA}.load_log'   # one row per load per run
STATUS_JSON           = '{"labels":{"Run_Audit_Log_Ingester":"Copilot audit log","Run_Audit_Log_Processor":"Copilot usage (audit log processing)","Refresh_Semantic_Model":"ValueLens model refresh","Run_Licensed_Users_Ingester":"Licensed users","Run_Upload_Router":"Uploaded CSV exports","Run_Agent365_Registry_Ingester":"Agent 365 (API)","Run_Agent365_CSV_Fallback":"Agent 365 (export)","Run_Agent365_Lander":"Agent 365 (export)","Run_Org_Data_Ingester":"Org data (Entra ID)","Run_Org_Data_Workday":"Org data (Workday export)","Run_Access_Scope":"Who sees which orgs (org-scoped access)","Run_M365_Activity_Ingester":"Microsoft 365 activity","Run_ProductFeedback_Ingester":"Product feedback","Run_Resource_Graph_Ingester":"Agent configuration and Foundry (Resource Graph)","Run_Defender_Ingester":"Shadow AI and agent risk (Defender)","Record_Load_Status":"Load status record"},"reasons":[{"kind":"capacity","pattern":"TooManyRequestsForCapacity|(?:code|status)[\\\\s:\'\\"=-]{0,6}430\\\\b","text":"The capacity was too busy to start it."},{"kind":"signIn","pattern":"AADSTS\\\\d+|\\\\b401\\\\b|\\\\b403\\\\b|Unauthori[sz]ed|Forbidden|invalid_client|consent|credential|sign.?in","text":"It couldn\'t sign in, or wasn\'t allowed to read the data."},{"kind":"timeout","pattern":"timed? ?out|timeout|exceeded the (?:maximum|allowed)","text":"It ran out of time."},{"kind":"noData","pattern":"PATH_NOT_FOUND|Path does not exist|FileNotFound|No such file","text":"The file it reads wasn\'t there."}],"statusTables":[{"activity":"Run_Resource_Graph_Ingester","table":"arg_status","item":"Probe","status":"Status"},{"activity":"Run_Defender_Ingester","table":"defender_status","item":"Probe","status":"Status"}],"warnings":[{"kind":"forbidden","statuses":["forbidden"],"all":false,"text":"It ran, but wasn\'t allowed to read {items}."},{"kind":"unlicensed","statuses":["unlicensed"],"all":false,"text":"It ran, but the tenant isn\'t licensed for {items}."},{"kind":"other","statuses":["error"],"all":false,"text":"It ran, but couldn\'t read {items}."},{"kind":"empty","statuses":["empty"],"all":true,"text":"It ran, but found nothing to load in {items}."}]}'

In [ ]:
# === HELPERS (pure Python helpers) ===
import json
import re
from datetime import datetime, timedelta, timezone

SELF = 'Record_Load_Status'
REGISTRY = 'Run_Agent365_Registry_Ingester'
FALLBACK = 'Run_Agent365_CSV_Fallback'
FAILED = ('Failed', 'Cancelled', 'TimedOut')
GUID = re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')


def label(name, labels):
    if name in labels:
        return labels[name]
    text = re.sub(r'^Run_', '', name)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    return text.replace('_', ' ')


def classify(error, reasons, activity=None):
    # Same order and patterns as the installer's loads.js, so both say the same thing.
    # A reason tied to one step (such as the Cowork Dataflow's sign-in) matches only that step.
    error = error or {}
    subject = f"errorCode={error.get('errorCode') or ''} {error.get('message') or ''}"
    for r in reasons:
        if r.get('activity') and r['activity'] != activity:
            continue
        if re.search(r['pattern'], subject, re.IGNORECASE):
            return r['kind'], r['text']
    first = next((l.strip() for l in str(error.get('message') or '').splitlines() if l.strip()), '')
    return 'other', (f'It stopped with an error: {first[:240]}' if first else 'It stopped without saying why.')


def parse_time(text):
    # Pipeline times carry up to 7 fractional digits, which older Pythons can't parse; seconds are enough.
    m = re.match(r'^\s*(\d{4}-\d{2}-\d{2})[T ](\d{2}:\d{2}:\d{2})', str(text or ''))
    if not m:
        return None
    return datetime.strptime(f'{m.group(1)}T{m.group(2)}', '%Y-%m-%dT%H:%M:%S').replace(tzinfo=timezone.utc)


def window(trigger_time, now):
    start = parse_time(trigger_time) or now - timedelta(days=2)
    return start - timedelta(hours=1), now + timedelta(hours=1)


def last_attempts(runs):
    # The last attempt of each step, with how many attempts there were.
    last = {}
    for r in sorted(runs, key=lambda r: str(r.get('activityRunStart') or '')):
        name = r.get('activityName')
        prev = last.get(name)
        last[name] = dict(r, attempts=(prev['attempts'] if prev else 0) + 1)
    return last


def is_load(r):
    return r.get('activityName') != SELF and r.get('activityType') != 'IfCondition'


def failed_loads(last):
    status = {n: r.get('status') for n, r in last.items()}
    out = []
    for name, r in last.items():
        if not is_load(r) or r.get('status') not in FAILED:
            continue
        if name == REGISTRY and status.get(FALLBACK) == 'Succeeded':
            continue
        if name == FALLBACK and status.get(REGISTRY) == 'Succeeded':
            continue
        out.append(name)
    return out


def warning(status_rows, rules):
    # Why a load that succeeded is a warning, from its status table's rows ({'item', 'status'}), or None.
    seen = [(str(r.get('item') or '').strip(), str(r.get('status') or '').strip().lower()) for r in status_rows]
    for rule in rules:
        hits = [item for item, status in seen if status in rule['statuses']]
        if not hits:
            continue
        if rule.get('all') and any(status not in ('empty', 'skipped') for _, status in seen):
            continue
        items = ', '.join(dict.fromkeys(i for i in hits if i)) or 'its source'
        return rule['kind'], rule['text'].replace('{items}', items)
    return None


def rows(last, labels, reasons, run_id, logged_at, warnings=None):
    out = []
    for name, r in last.items():
        if not is_load(r):
            continue
        status = r.get('status')
        error = r.get('error') or {}
        kind, reason = classify(error, reasons, name) if status in FAILED else (None, None)
        if status == 'Succeeded' and warnings and name in warnings:
            status = 'Warning'
            kind, reason = warnings[name]
        ms = r.get('durationInMs')
        out.append({
            'run_id': run_id,
            'activity': name,
            'source': label(name, labels),
            'activity_type': r.get('activityType'),
            'status': status,
            'reason_kind': kind,
            'reason': reason,
            'attempts': int(r.get('attempts') or 1),
            'started_at': parse_time(r.get('activityRunStart')),
            'ended_at': parse_time(r.get('activityRunEnd')),
            'duration_seconds': None if ms is None else round(float(ms) / 1000, 1),
            'error_code': (str(error.get('errorCode')) or None) if error.get('errorCode') else None,
            'error_message': (str(error.get('message'))[:4000] or None) if error.get('message') else None,
            'logged_at': logged_at,
        })
    return out

In [ ]:

import json
import re
from datetime import datetime, timezone
from pyspark.sql import types as T
from valuelens_databricks_runtime import workflow_task_runs

STATUS = json.loads(STATUS_JSON)
JOB_RUN_ID = VL.widget("job_run_id", "").strip()
TASK_STATES_JSON = VL.widget("task_states_json", "{}")
LOG_TABLE = f"{DB_SCHEMA}.load_log"
NOW = datetime.now(timezone.utc)
SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("activity", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("activity_type", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("reason_kind", T.StringType()),
    T.StructField("reason", T.StringType()),
    T.StructField("attempts", T.IntegerType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("ended_at", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("error_code", T.StringType()),
    T.StructField("error_message", T.StringType()),
    T.StructField("logged_at", T.TimestampType()),
])
REQUIRED_TASKS = {
    "Run_Audit_Log_Ingester",
    "Run_Audit_Log_Processor",
    "Run_Licensed_Users_Ingester",
    "Run_Org_Data_Ingester",
    "Refresh_Semantic_Model",
}
TASK_VALUE_ALIASES = {
    "Run_Agent365_CSV_Fallback": "Run_Agent365_Lander",
}


def _workflow_task_runs():
    if not JOB_RUN_ID:
        return {}
    if not re.fullmatch(r"\d+", JOB_RUN_ID):
        raise ValueError("job_run_id must be the numeric Databricks workflow run ID.")
    raw_states = json.loads(TASK_STATES_JSON or "{}")
    task_values = {}
    for task_key in STATUS["labels"]:
        task_value_key = TASK_VALUE_ALIASES.get(task_key, task_key)
        if task_key in raw_states or task_value_key in raw_states:
            task_values[task_value_key] = VL.task_status(task_value_key) or {}
    return workflow_task_runs(
        STATUS["labels"],
        raw_states,
        task_values,
        REQUIRED_TASKS,
        TASK_VALUE_ALIASES,
    )


def record():
    if not JOB_RUN_ID:
        print("No Databricks workflow run ID; load status was not recorded.")
        return []

    last = _workflow_task_runs()
    warnings = find_warnings(last)
    data = rows(
        last,
        STATUS["labels"],
        STATUS["reasons"],
        JOB_RUN_ID,
        NOW.replace(tzinfo=None),
        warnings,
    )
    for row in data:
        for field in ("started_at", "ended_at"):
            if row[field] is not None:
                row[field] = row[field].replace(tzinfo=None)
    if not data:
        print(f"No load-status rows were produced for workflow run {JOB_RUN_ID}.")
        return []

    if not spark.catalog.tableExists(LOG_TABLE):
        spark.createDataFrame(data, schema=SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)
    else:
        spark.createDataFrame(data, schema=SCHEMA).createOrReplaceTempView("_valuelens_load_status")
        spark.sql(
            f"DELETE FROM {LOG_TABLE} "
            f"WHERE run_id = '{JOB_RUN_ID}'"
        )
        spark.sql(
            f"INSERT INTO {LOG_TABLE} "
            "SELECT * FROM _valuelens_load_status"
        )

    failures = failed_loads(last)
    if failures:
        raise RuntimeError(f"ValueLens workflow completed with failed tasks: {failures}")
    print(f"Recorded {len(data)} load-status rows for workflow run {JOB_RUN_ID}.")
    return []


record()
dbutils.notebook.exit("Succeeded")
